In [6]:
!pip install -q unsloth trl peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.

In [7]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
from transformers import BitsAndBytesConfig

max_seq_length = 4096
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,       # double quant saves ~0.4GB
    bnb_4bit_compute_dtype = dtype,          # upcast to bf16/fp16 for forward pass
)

In [9]:
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3.5-2B",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = True,
        quantization_config = bnb_cfg,
    )
    print("loaded unsloth 4bit version")
except Exception as e:
    print(f"unsloth 4bit not found: {e}")
    print("falling back to base HF model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "Qwen/Qwen3.5-2B",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = True,
        quantization_config = bnb_cfg,
    )
    print("loaded")

==((====))==  Unsloth 2026.3.7: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

loaded unsloth 4bit version


In [10]:
from datasets import load_dataset

# keep_in_memory=False = memory-mapped Arrow file on disk, not loaded into RAM
ds = load_dataset(
    "nohurry/Opus-4.6-Reasoning-3000x-filtered",
    split="train",
    keep_in_memory=False,
)
print(ds)
print("\nColumns:", ds.column_names)
for k, v in ds[0].items():
    print(f"\n--- {k} ---\n{str(v)[:400]}")

README.md:   0%|          | 0.00/185 [00:00<?, ?B/s]

(…)lled_corpus_400k_with_cot-filtered.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash'],
    num_rows: 2326
})

Columns: ['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash']

--- id ---
orca_3452

--- problem ---
252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?

--- thinking ---
Let me work through this problem step by step.

First, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?

Key values given: 252, 8, 41, 300,000, 7,500, ,

My approach:
1. F

--- solution ---
# Solution: Calculating Bus Rental and Toll Costs

#

In [11]:
sample = ds[0]
cols = list(sample.keys())

if "prompt" in cols and "response" in cols:
    usr_col, asst_col = "prompt", "response"
elif "prompt" in cols and "completion" in cols:
    usr_col, asst_col = "prompt", "completion"
elif "question" in cols and "answer" in cols:
    usr_col, asst_col = "question", "answer"
elif "instruction" in cols and "output" in cols:
    usr_col, asst_col = "instruction", "output"
else:
    usr_col, asst_col = cols[0], cols[1]

print(f"user col: '{usr_col}'  |  assistant col: '{asst_col}'")
print(f"\nSample response (first 300 chars):\n{str(sample[asst_col])[:300]}")

user col: 'id'  |  assistant col: 'problem'

Sample response (first 300 chars):
252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?


In [12]:
import gc
EOS = tokenizer.eos_token

def fmt(examples):
    texts = []
    for u, a in zip(examples[usr_col], examples[asst_col]):
        msg = [
            {"role": "user",      "content": str(u)},
            {"role": "assistant", "content": str(a)},
        ]
        text = tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=False,
        ) + EOS
        texts.append(text)
    return {"text": texts}

ds_fmt = ds.map(
    fmt,
    batched=True,
    batch_size=50,                        # small batch = lower RAM spike during map
    remove_columns=ds.column_names,       # drop ALL original columns immediately during map
    num_proc=1,
)

# kill the original dataset from RAM immediately
del ds
gc.collect()

# flatten compacts the arrow file on disk, removes fragmentation
ds_fmt = ds_fmt.flatten_indices()

print(f"dataset size in memory: {ds_fmt.dataset_size / 1024**2:.1f} MB")
print(f"sample (first 400 chars):\n{ds_fmt[0]['text'][:400]}")

Map (num_proc=1):   0%|          | 0/2326 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/2326 [00:00<?, ? examples/s]

dataset size in memory: 6.8 MB
sample (first 400 chars):
<|im_start|>user
orca_3452<|im_end|>
<|im_start|>assistant
<think>

</think>

252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?<|im_end|>
<|im_end|>


In [13]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,              # bumped to 2x r — common QLoRA practice
    lora_dropout = 0.05,          # small dropout helps with 3k sample overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()
# should show ~0.5% trainable — everything else frozen in 4bit NF4

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients
trainable params: 10,911,744 || all params: 2,224,153,408 || trainable%: 0.4906


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments
import gc
import torch

# Clear memory before initialization
gc.collect()
torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_fmt,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataloader_num_workers = 0,
        dataloader_pin_memory = False,
        max_grad_norm = 0.3
    ),
)
print("Trainer initialized successfully.")

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2326 [00:00<?, ? examples/s]

Trainer initialized successfully.


In [15]:
import sys
import torch
import gc

# Increase recursion limit to prevent bitsandbytes depth errors
sys.setrecursionlimit(3000)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Starting training...")
trainer_stats = trainer.train()

peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak VRAM used: {peak:.2f} GB")
print(f"Training Stats: {trainer_stats}")

Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,326 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Step,Training Loss
1,3.271208
2,3.072265
3,3.237006
4,3.152545
5,2.740386
6,2.221735
7,2.713271
8,2.809144
9,2.372698
10,2.317172


Peak VRAM used: 10.81 GB
Training Stats: TrainOutput(global_step=60, training_loss=2.1720685601234435, metrics={'train_runtime': 616.1411, 'train_samples_per_second': 0.779, 'train_steps_per_second': 0.097, 'total_flos': 611665924098048.0, 'train_loss': 2.1720685601234435, 'epoch': 0.2063628546861565})


In [17]:
from google.colab import userdata
from huggingface_hub import login

# 1. Fetch token and authenticate the session globally
# This prevents underlying git/tokenizer errors that sometimes ignore the 'token=' kwarg
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

# Define your repo name once to avoid typos
repo_id = "Srikri7/qwen3.5-2b-reasoning"
local_dir = "qwen35_2b_reasoning"

# 2. Save locally
model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)

# 3. Push to Hugging Face Hub (login() handles the token now)
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

# 4. Fixed the URL (your original print statement was missing the "." in 3.5)
print(f"Pushed to: https://huggingface.co/{repo_id}")

README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  559kB / 43.7MB            

Saved model to https://huggingface.co/Srikri7/qwen3.5-2b-reasoning


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpez66ethj/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Pushed to: https://huggingface.co/Srikri7/qwen3.5-2b-reasoning
